In [ ]:
# ###############################################
# SCRIPT: Fornecedor Principal - Divergências
# DATA DE CRIAÇÃO: 11/07/2025
#
# Objetivo:
# O código identifica divergências de fornecedores entre dois dias consecutivos
# no "mapa de estoque" (dados vindos do SQL Server).
# Ele compara o fornecedor principal (COD_FORNECEDOR) dos itens e identifica
# quando houve mudança (ou novo cadastro de item).
# ###############################################

print("######################### FORNECEDOR PRINCIPAL - DIVERGÊNCIAS #########################\n")

# ---- Importações de bibliotecas ----
import os                 # Trabalha com caminhos e arquivos no sistema operacional
import urllib             # Usado para formatar parâmetros de conexão com o banco
import datetime           # Manipula datas
import pandas as pd       # Manipulação de dados (DataFrames)
from sqlalchemy import create_engine, text  # Conecta e executa consultas SQL

import warnings           # Suprime avisos desnecessários
warnings.filterwarnings('ignore')  # Ignora warnings no console


# ---- Definição de caminhos ----
upload_ftp_folder_path = r"S:\Tributario\Preços\Upload FTP\Upload"  # Caminho para upload automático (FTP)
folder_path = r"S:\Tributario\Preços\01. HB\02. Automações folhas preços\YDAI - Fornecedor Principal"  # Backup local


# ---- Configurações de conexão com o banco SQL Server ----
server_name = 'LST_IMGRUPOSC'
username = 'im_preco'
password = 'Pr4949d#85'  # (⚠️ ideal seria usar variável de ambiente em vez de deixar exposto)

# Monta a string de conexão de forma segura (usando urllib)
params = urllib.parse.quote_plus(
    "Driver={SQL Server};"
    "Server=" + server_name + ";"
    "uid=" + username + ";"
    "pwd=" + password
)

# Cria o motor de conexão SQLAlchemy (com suporte a pyodbc)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", use_setinputsizes=False)

print('Lendo dados do lake...\n')


# ---- Consulta SQL 1: Data mais recente ----
# Busca o fornecedor principal mais recente (último dia disponível no banco)
query = """
SELECT ORDERDATE AS DATA,
       DES_CD AS CD,
       COD_MATERIAL_SAP AS SAP,
       MAX(COD_FORNECEDOR) AS 'COD_FORNECEDOR_D0'
FROM OURO..v_IM_PRECO_PRODUTO_CD_DIARIO
WHERE ORDERDATE = (SELECT MAX(ORDERDATE) FROM OURO..v_IM_PRECO_PRODUTO_CD_DIARIO)
  AND COD_MATERIAL_SAP LIKE '7%'       -- filtra apenas materiais que começam com 7
  AND COD_FORNECEDOR <> ''             -- exclui fornecedores vazios
  AND COD_FORNECEDOR IS NOT NULL
GROUP BY COD_MATERIAL_SAP, DES_CD, ORDERDATE;
"""

# Lê os dados da query e converte em DataFrame do pandas
df_mapa_d0 = pd.read_sql_query(sql=text(query), con=engine.connect())

# Limpa e ajusta campos
df_mapa_d0['SAP'] = df_mapa_d0['SAP'].astype(str).str.replace('.0', '')  # remove sufixo ".0" dos códigos
df_mapa_d0['COD_FORNECEDOR_D0'] = df_mapa_d0['COD_FORNECEDOR_D0'].str.lstrip('0')  # remove zeros à esquerda


# ---- Consulta SQL 2: Segunda data mais recente ----
# Busca o fornecedor principal do dia anterior ao mais recente
query = """
SELECT ORDERDATE AS DATA,
       DES_CD AS CD,
       COD_MATERIAL_SAP AS SAP,
       MAX(COD_FORNECEDOR) AS 'COD_FORNECEDOR_DM1'
FROM OURO..v_IM_PRECO_PRODUTO_CD_DIARIO
WHERE ORDERDATE = (
         SELECT TOP 1 ORDERDATE
         FROM OURO..v_IM_PRECO_PRODUTO_CD_DIARIO
         WHERE ORDERDATE NOT IN (SELECT MAX(ORDERDATE) FROM OURO..v_IM_PRECO_PRODUTO_CD_DIARIO)
         ORDER BY ORDERDATE DESC
     )
  AND COD_MATERIAL_SAP LIKE '7%'
  AND COD_FORNECEDOR <> ''
  AND COD_FORNECEDOR IS NOT NULL
GROUP BY COD_MATERIAL_SAP, DES_CD, ORDERDATE;
"""

# Executa a query e obtém o DataFrame do dia anterior
df_mapa_dm1 = pd.read_sql_query(sql=text(query), con=engine.connect())

# Limpeza dos dados (mesmo processo do anterior)
df_mapa_dm1['SAP'] = df_mapa_dm1['SAP'].astype(str).str.replace('.0', '')
df_mapa_dm1['COD_FORNECEDOR_DM1'] = df_mapa_dm1['COD_FORNECEDOR_DM1'].str.lstrip('0')


# ---- Cruzamento entre as duas bases ----
# Une os dois DataFrames pelo mesmo material e centro de compra (CD)
df_mapa_div = (
    df_mapa_d0.merge(
        df_mapa_dm1,
        on=['CD', 'SAP'],          # chaves para o merge
        how='outer',               # outer = mantém todos os registros (mesmo que não coincidam)
        suffixes=['_D0', '_DM1']   # sufixos para diferenciar colunas iguais
    )
    # Filtra os casos em que o fornecedor antigo não existe (novos cadastros)
    .query("COD_FORNECEDOR_DM1.isna()")
    # Renomeia colunas para um formato mais amigável
    .rename(columns={
        'SAP': 'Material',
        'CD': 'Centro Compra',
        'COD_FORNECEDOR_D0': 'Fornecedor principal',
        'DATA_D0': 'Data inicial'
    })
)


# ---- Se houver divergências ----
if df_mapa_div.shape[0] != 0:  # shape[0] = número de linhas
    print(f"\tHá {df_mapa_div.shape[0]} códigos de fornecedor divergentes entre {df_mapa_dm1['DATA'].iloc[0]} e {df_mapa_d0['DATA'].iloc[0]}.")

    # Cria colunas adicionais exigidas pelo sistema de preços (condições SAP)
    df_mapa_div['Data inicial'] = datetime.datetime.today().strftime("%d.%m.%Y")
    df_mapa_div['Aplicação'] = 'M'
    df_mapa_div['Tabela de cond'] = '587'
    df_mapa_div['Tipo Cond'] = 'YDAI'
    df_mapa_div['Data final'] = '31.12.9999'

    # Reordena as colunas no formato esperado
    df_mapa_div = df_mapa_div[[
        'Aplicação',
        'Tabela de cond',
        'Tipo Cond',
        'Material',
        'Centro Compra',
        'Fornecedor principal',
        'Data inicial',
        'Data final'
    ]]

    # Salva os resultados em dois locais:
    # 1️⃣ Envia para a pasta de upload FTP
    df_mapa_div.to_csv(
        os.path.join(upload_ftp_folder_path, f"YDAI_{datetime.datetime.today().strftime('%Y%m%d')}.csv"),
        index=False,
        sep=';'
    )

    # 2️⃣ Faz backup local na pasta "Folhas"
    df_mapa_div.to_csv(
        os.path.join(folder_path, "Folhas", f"YDAI_{datetime.datetime.today().strftime('%Y%m%d')}.csv"),
        index=False,
        sep=';',
        encoding='latin1'
    )

# ---- Caso não haja divergências ----
else:
    print(f"\tNão há registros divergentes entre {df_mapa_dm1['DATA'].iloc[0]} e {df_mapa_d0['DATA'].iloc[0]}.")